# Image Analysis — Azure AI Vision

This notebook uses **Azure AI Vision Image Analysis 4.0** to analyze an image and extract:

- **Caption** — a human-readable sentence describing the image *(only in select regions: East US, West US, West US 2, France Central, North Europe, West Europe, Sweden Central, Switzerland North, Australia East, Southeast Asia, East Asia, Korea Central, Japan East)*
- **Dense captions** — captions for individual regions *(same regional limitation)*
- **Objects** — bounding boxes and labels for detected objects
- **Tags** — topical labels for the image content
- **People** — bounding boxes for detected people

In [ ]:
%pip install azure-ai-vision-imageanalysis azure-core python-dotenv --quiet

In [ ]:
import os
from azure.ai.vision.imageanalysis import ImageAnalysisClient
from azure.ai.vision.imageanalysis.models import VisualFeatures
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # loads .env from repo root

endpoint = os.environ["AZURE_VISION_ENDPOINT"]
api_key = os.environ["AZURE_VISION_KEY"]

client = ImageAnalysisClient(endpoint=endpoint, credential=AzureKeyCredential(api_key))
print("Vision client ready.")

In [ ]:
# Use a publicly accessible sample image
# Replace IMAGE_URL with a URL to any image you want to analyze
IMAGE_URL = "https://learn.microsoft.com/azure/ai-services/computer-vision/media/quickstarts/presentation.png"

print(f"Analyzing image: {IMAGE_URL}")

# Request only the universally available features first
base_features = [
    VisualFeatures.OBJECTS,
    VisualFeatures.TAGS,
    VisualFeatures.PEOPLE,
    VisualFeatures.READ,
    VisualFeatures.SMART_CROPS,
]

# CAPTION and DENSE_CAPTIONS are only supported in a subset of regions.
# Try to include them, fall back gracefully if the region does not support them.
from azure.core.exceptions import HttpResponseError

try:
    result = client.analyze_from_url(
        image_url=IMAGE_URL,
        visual_features=base_features + [VisualFeatures.CAPTION, VisualFeatures.DENSE_CAPTIONS],
        language="en",
    )
    print("Analysis complete (with captions).")
except HttpResponseError as exc:
    if "not supported in this region" in str(exc):
        print("Captions not available in this region — re-running without them.")
        result = client.analyze_from_url(
            image_url=IMAGE_URL,
            visual_features=base_features,
            language="en",
        )
        print("Analysis complete (no captions).")
    else:
        raise

In [ ]:
# Display caption (only if available in this region)
if result.caption:
    print(f"Caption: {result.caption.text} (confidence: {result.caption.confidence:.4f})")
else:
    print("No caption available (feature not supported in this region).")

In [ ]:
# Display detected objects
if result.objects and result.objects.list:
    print("\nDetected objects:")
    for obj in result.objects.list:
        r = obj.bounding_box
        tags = ", ".join(t.name for t in obj.tags) if obj.tags else "(no tags)"
        print(f"  {tags} @ [{r.x}, {r.y}, {r.width}x{r.height}]")

In [ ]:
# Display image tags
if result.tags and result.tags.list:
    print("\nImage tags:")
    for tag in result.tags.list:
        print(f"  {tag.name} (confidence: {tag.confidence:.4f})")

In [ ]:
# Display dense captions (region-level descriptions) — only if available
if result.dense_captions and result.dense_captions.list:
    print("\nDense captions (region descriptions):")
    for dc in result.dense_captions.list:
        r = dc.bounding_box
        print(f"  '{dc.text}' @ [{r.x}, {r.y}, {r.width}x{r.height}] (confidence: {dc.confidence:.4f})")
else:
    print("No dense captions available (feature not supported in this region).")